#**Generating Your First Text**

In [ ]:
!pip install -q transformers==4.48.3 accelerate sentencepiece
import transformers
print(transformers.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 66.4 MB/s eta 0:00:00
4.48.3


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
#Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=True,
    revision="main"
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct", revision="main")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

modeling_phi3.py: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [ ]:
from transformers import pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False,
    temperature=0.7
)

Device set to use cuda


In [ ]:
# The prompt
messages = [
    {"role": "user",
     "content": "Create a funny joke about India."
     }
]

# Generate output
output = generator(messages)
print(output[0]["generated_text"])

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Calling `get_max_cache()` will raise error from v4.48


 Why don't secrets stay secret in India? Because everyone has a "spice" for gossip!


# **Tokenizer**

In [ ]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened. <|assistance|>"

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# Generate the text
generation_output = model.generate(
    input_ids=input_ids,
    max_new_tokens=20
)

# Print the output
print(tokenizer.decode(generation_output[0]))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened. <|assistance|> Dear Sarah,

I hope this message finds you well. I am writing to express my


In [ ]:
# Prompt's/Input token IDs
print(input_ids)

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889,   529, 29989,   465, 21558, 29989, 29958]],
       device='cuda:0')


In [ ]:
# Decode the prompt/Input from their token ids.
for id in input_ids[0]:
  print(tokenizer.decode(id))

Write
an
email
apolog
izing
to
Sarah
for
the
trag
ic
garden
ing
m
ish
ap
.
Exp
lain
how
it
happened
.
<
|
ass
istance
|
>


In [ ]:
# input tokens as well as the output tokens ids.
print(generation_output)

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889,   529, 29989,   465, 21558, 29989, 29958,   360,
           799, 19235, 29892,    13,    13, 29902,  4966,   445,  2643, 14061,
           366,  1532, 29889,   306,   626,  5007,   304,  4653,   590]],
       device='cuda:0')


In [ ]:
print(tokenizer.decode(16423))
print(tokenizer.decode(292))
print(tokenizer.decode([16423,292]))

garden
ing
gardening


#**Comparing Trained LLM Tokenizers**


### 1. BERT Tokenizer (WordPiece)

In [ ]:
# !pip install transformers

from transformers import AutoTokenizer

# Load the BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Test sentence
text = text = """
Hello     ChatGPT!

I love NLP 😊.
Email: training123@gmail.com

Price = $45.67
Version=v2.5.1
snake_case_variable
camelCaseVariable
C++, Python, Java!
भारत 日本 العربية
"""

print("=" * 60)
print("Original Text:")
print(text)

# Tokenize
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("\n" + "=" * 60)
print("Tokens:")
for i, token in enumerate(tokens):
    print(f"{i:2d}: {repr(token)}")

print("\n" + "=" * 60)
print("Token IDs:")
print(token_ids)

print("\n" + "=" * 60)
print("Token -> ID Mapping:")
for token, tid in zip(tokens, token_ids):
    print(f"{repr(token):20} -> {tid}")

# Encode with special tokens
encoded = tokenizer.encode(text, add_special_tokens=True)

print("\n" + "=" * 60)
print("Encoded IDs (with special tokens):")
print(encoded)

print("\nDecoded back:")
print(tokenizer.decode(encoded))

print("\n" + "=" * 60)
print("Special Tokens:")
print("CLS Token :", tokenizer.cls_token)
print("SEP Token :", tokenizer.sep_token)
print("PAD Token :", tokenizer.pad_token)
print("UNK Token :", tokenizer.unk_token)
print("MASK Token:", tokenizer.mask_token)

Original Text:

Hello     ChatGPT!

I love NLP 😊.
Email: training123@gmail.com

Price = $45.67
Version=v2.5.1
snake_case_variable
camelCaseVariable
C++, Python, Java!
भारत 日本 العربية


Tokens:
 0: 'hello'
 1: 'chat'
 2: '##gp'
 3: '##t'
 4: '!'
 5: 'i'
 6: 'love'
 7: 'nl'
 8: '##p'
 9: '[UNK]'
10: '.'
11: 'email'
12: ':'
13: 'training'
14: '##12'
15: '##3'
16: '@'
17: 'gma'
18: '##il'
19: '.'
20: 'com'
21: 'price'
22: '='
23: '$'
24: '45'
25: '.'
26: '67'
27: 'version'
28: '='
29: 'v'
30: '##2'
31: '.'
32: '5'
33: '.'
34: '1'
35: 'snake'
36: '_'
37: 'case'
38: '_'
39: 'variable'
40: 'camel'
41: '##case'
42: '##var'
43: '##iable'
44: 'c'
45: '+'
46: '+'
47: ','
48: 'python'
49: ','
50: 'java'
51: '!'
52: 'भ'
53: '##ा'
54: '##र'
55: '##त'
56: '日'
57: '本'
58: 'ا'
59: '##ل'
60: '##ع'
61: '##ر'
62: '##ب'
63: '##ي'
64: '##ة'

Token IDs:
[7592, 11834, 21600, 2102, 999, 1045, 2293, 17953, 2361, 100, 1012, 10373, 1024, 2731, 12521, 2509, 1030, 20917, 4014, 1012, 4012, 3976, 1027, 1002, 3429, 10

###2. GPT Tokenizer (using tiktoken)

In [ ]:
# !pip install tiktoken

import tiktoken

# Load GPT tokenizer (used by many OpenAI models)
encoding = tiktoken.get_encoding("cl100k_base")

text = """
Hello, world!     This is a tokenizer test.

Email: john.doe@example.com
Price: $19.99
Emoji: 😊
Python_version = 3.11
C++ > Java?
"""

print("=" * 60)
print("Original Text:")
print(text)

# Encode
token_ids = encoding.encode(text)

print("\n" + "=" * 60)
print("Token IDs:")
print(token_ids)

print("\n" + "=" * 60)
print("Token Breakdown:")

for i, token_id in enumerate(token_ids):
    token_bytes = encoding.decode_single_token_bytes(token_id)

    try:
        token_text = token_bytes.decode("utf-8")
    except UnicodeDecodeError:
        token_text = str(token_bytes)

    print(f"{i:2d}: ID={token_id:<8} Token={repr(token_text)}")

print("\n" + "=" * 60)
print("Decoded Text:")
print(encoding.decode(token_ids))

print("\n" + "=" * 60)
print("Total Tokens:", len(token_ids))

Original Text:

Hello, world!     This is a tokenizer test.

Email: john.doe@example.com
Price: $19.99
Emoji: 😊
Python_version = 3.11
C++ > Java?


Token IDs:
[198, 9906, 11, 1917, 0, 257, 1115, 374, 264, 47058, 1296, 382, 4886, 25, 40742, 962, 4748, 36587, 916, 198, 7117, 25, 400, 777, 13, 1484, 198, 93831, 25, 27623, 232, 198, 31380, 9625, 284, 220, 18, 13, 806, 198, 34, 1044, 871, 8102, 5380]

Token Breakdown:
 0: ID=198      Token='\n'
 1: ID=9906     Token='Hello'
 2: ID=11       Token=','
 3: ID=1917     Token=' world'
 4: ID=0        Token='!'
 5: ID=257      Token='    '
 6: ID=1115     Token=' This'
 7: ID=374      Token=' is'
 8: ID=264      Token=' a'
 9: ID=47058    Token=' tokenizer'
10: ID=1296     Token=' test'
11: ID=382      Token='.\n\n'
12: ID=4886     Token='Email'
13: ID=25       Token=':'
14: ID=40742    Token=' john'
15: ID=962      Token='.d'
16: ID=4748     Token='oe'
17: ID=36587    Token='@example'
18: ID=916      Token='.com'
19: ID=198      Token='\n'
20: I

###3. StarCoder Tokenizer
Unlike BERT (optimized for natural language) and GPT's general-purpose tokenizer, StarCoder's tokenizer is trained heavily on source code, so it often treats code patterns, operators, indentation, and identifiers more naturally.

In [ ]:
!pip install transformers sentencepiece

from transformers import AutoTokenizer
import huggingface_hub
huggingface_hub.login(token="hf_access_token") # Replace the hf_access_token

# Load StarCoder tokenizer
tokenizer = AutoTokenizer.from_pretrained("bigcode/starcoder")

# Test text containing both natural language and code
text = text = """
for i in range(10):
    print(i)

my_variable = "Hello, World!"
total_price = 45.67

if x >= 100 and y != 0:
    result += x // y

# Comment
user_email = "training123@gmail.com"

C++
Python3.11
snake_case
camelCase
😊
"""

print("=" * 60)
print("Original Text:")
print(text)

# Tokenize
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("\n" + "=" * 60)
print("Tokens:")
for i, token in enumerate(tokens):
    print(f"{i:2d}: {repr(token)}")

print("\n" + "=" * 60)
print("Token IDs:")
print(token_ids)

print("\n" + "=" * 60)
print("Token -> ID Mapping:")
for token, tid in zip(tokens, token_ids):
    print(f"{repr(token):30} -> {tid}")

# Encode
encoded = tokenizer.encode(text)

print("\n" + "=" * 60)
print("Encoded IDs:")
print(encoded)

print("\nDecoded back:")
print(tokenizer.decode(encoded))

print("\n" + "=" * 60)
print("Total Tokens:", len(tokens))

Original Text:

for i in range(10):
    print(i)

my_variable = "Hello, World!"
total_price = 45.67

if x >= 100 and y != 0:
    result += x // y

# Comment
user_email = "training123@gmail.com"

C++
Python3.11
snake_case
camelCase
😊


Tokens:
 0: 'Ċ'
 1: 'for'
 2: 'Ġi'
 3: 'Ġin'
 4: 'Ġrange'
 5: '('
 6: '1'
 7: '0'
 8: '):'
 9: 'ĊĠĠĠ'
10: 'Ġprint'
11: '('
12: 'i'
13: ')'
14: 'Ċ'
15: 'Ċ'
16: 'my'
17: '_'
18: 'variable'
19: 'Ġ='
20: 'Ġ"'
21: 'Hello'
22: ','
23: 'ĠWorld'
24: '!"'
25: 'Ċ'
26: 'total'
27: '_'
28: 'price'
29: 'Ġ='
30: 'Ġ'
31: '4'
32: '5'
33: '.'
34: '6'
35: '7'
36: 'Ċ'
37: 'Ċ'
38: 'if'
39: 'Ġx'
40: 'Ġ>='
41: 'Ġ'
42: '1'
43: '0'
44: '0'
45: 'Ġand'
46: 'Ġy'
47: 'Ġ!='
48: 'Ġ'
49: '0'
50: ':'
51: 'ĊĠĠĠ'
52: 'Ġresult'
53: 'Ġ+='
54: 'Ġx'
55: 'Ġ//'
56: 'Ġy'
57: 'Ċ'
58: 'Ċ'
59: '#'
60: 'ĠComment'
61: 'Ċ'
62: 'user'
63: '_'
64: 'email'
65: 'Ġ='
66: 'Ġ"'
67: 'training'
68: '1'
69: '2'
70: '3'
71: '@'
72: 'gmail'
73: '.'
74: 'com'
75: '"'
76: 'Ċ'
77: 'Ċ'
78: 'C'
79: '++'
80: 'Ċ'
81: 'P

###4. Llama Tokenizer (SentencePiece)

In [ ]:
!pip install transformers sentencepiece

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

text = """
Hello, world!

Email: john.doe@example.com
Price: $19.99
Emoji: 😊
Python_version = 3.11
C++ > Java?
"""

tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("=" * 60)
print("Original Text:")
print(text)

print("\nTokens:")
for i, token in enumerate(tokens):
    print(f"{i:2d}: {repr(token)}")

print("\nToken -> ID Mapping:")
for token, tid in zip(tokens, token_ids):
    print(f"{repr(token):25} -> {tid}")

print("\nDecoded:")
print(tokenizer.decode(token_ids))

Original Text:

Hello, world!

Email: john.doe@example.com
Price: $19.99
Emoji: 😊
Python_version = 3.11
C++ > Java?


Tokens:
 0: '▁'
 1: '<0x0A>'
 2: 'Hello'
 3: ','
 4: '▁world'
 5: '!'
 6: '<0x0A>'
 7: '<0x0A>'
 8: 'Email'
 9: ':'
10: '▁j'
11: 'ohn'
12: '.'
13: 'do'
14: 'e'
15: '@'
16: 'example'
17: '.'
18: 'com'
19: '<0x0A>'
20: 'Price'
21: ':'
22: '▁$'
23: '1'
24: '9'
25: '.'
26: '9'
27: '9'
28: '<0x0A>'
29: 'E'
30: 'mo'
31: 'ji'
32: ':'
33: '▁'
34: '<0xF0>'
35: '<0x9F>'
36: '<0x98>'
37: '<0x8A>'
38: '<0x0A>'
39: 'Python'
40: '_'
41: 'version'
42: '▁='
43: '▁'
44: '3'
45: '.'
46: '1'
47: '1'
48: '<0x0A>'
49: 'C'
50: '++'
51: '▁>'
52: '▁Java'
53: '?'
54: '<0x0A>'

Token -> ID Mapping:
'▁'                       -> 29871
'<0x0A>'                  -> 13
'Hello'                   -> 10994
','                       -> 29892
'▁world'                  -> 3186
'!'                       -> 29991
'<0x0A>'                  -> 13
'<0x0A>'                  -> 13
'Email'                   -> 982

###5. Qwen Tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-0.5B",
    trust_remote_code=True
)

text = """
Hello     ChatGPT!

Email: training123@gmail.com
Price = $45.67

snake_case_variable
camelCaseVariable

C++ >= Java?
😊

भारत 日本 العربية
"""

print("=" * 60)
print("Original Text:")
print(text)

# Tokenize
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("\n" + "=" * 60)
print("Tokens:")
for i, token in enumerate(tokens):
    print(f"{i:2d}: {repr(token)}")

print("\n" + "=" * 60)
print("Token -> ID Mapping:")
for token, tid in zip(tokens, token_ids):
    print(f"{repr(token):30} -> {tid}")

print("\n" + "=" * 60)
print("Decoded:")
print(tokenizer.decode(token_ids))

print("\nTotal Tokens:", len(token_ids))

Original Text:

Hello     ChatGPT!

Email: training123@gmail.com
Price = $45.67

snake_case_variable
camelCaseVariable

C++ >= Java?
😊

भारत 日本 العربية


Tokens:
 0: 'Ċ'
 1: 'Hello'
 2: 'ĠĠĠĠ'
 3: 'ĠChat'
 4: 'G'
 5: 'PT'
 6: '!ĊĊ'
 7: 'Email'
 8: ':'
 9: 'Ġtraining'
10: '1'
11: '2'
12: '3'
13: '@gmail'
14: '.com'
15: 'Ċ'
16: 'Price'
17: 'Ġ='
18: 'Ġ$'
19: '4'
20: '5'
21: '.'
22: '6'
23: '7'
24: 'ĊĊ'
25: 'snake'
26: '_case'
27: '_variable'
28: 'Ċ'
29: 'camel'
30: 'Case'
31: 'Variable'
32: 'ĊĊ'
33: 'C'
34: '++'
35: 'Ġ>='
36: 'ĠJava'
37: '?Ċ'
38: 'ðŁĺĬ'
39: 'ĊĊ'
40: 'à¤Ń'
41: 'à¤¾à¤'
42: '°'
43: 'à¤¤'
44: 'ĠæĹ¥'
45: 'æľ¬'
46: 'ĠØ§ÙĦØ¹Ø±Ø¨ÙĬØ©'
47: 'Ċ'

Token -> ID Mapping:
'Ċ'                            -> 198
'Hello'                        -> 9707
'ĠĠĠĠ'                         -> 257
'ĠChat'                        -> 12853
'G'                            -> 38
'PT'                           -> 2828
'!ĊĊ'                          -> 2219
'Email'                        -> 4781
':'         